# 02 · Run experiments

Run every policy across a sweep of creation costs and reproduce the headline figures inline.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
ROOT = ROOT if (ROOT / 'data').exists() else ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('project root:', ROOT)

In [ ]:
import importlib.util, pandas as pd
spec = importlib.util.spec_from_file_location('run_experiments', ROOT / 'scripts' / 'run_experiments.py')
rx = importlib.util.module_from_spec(spec); spec.loader.exec_module(rx)
faq = pd.read_csv(ROOT / 'data' / 'sample_faq_library.csv')
stream = pd.read_csv(ROOT / 'data' / 'sample_stream.csv')
summary, traj = rx.run_experiment(faq, stream, backend='tfidf', seed=0)
print('embedding backend:', summary['backend'].iloc[0])
summary[['policy','creation_cost','avg_loss','cum_mismatch_loss','n_created','create_rate']]

### Average total cost by policy and creation cost

In [ ]:
summary.pivot_table(index='policy', columns='creation_cost', values='avg_loss').round(3)

## Figures

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('make_plots', ROOT / 'scripts' / 'make_plots.py')
mp = importlib.util.module_from_spec(spec); spec.loader.exec_module(mp)
out = ROOT / 'results'; out.mkdir(exist_ok=True)
mp.plot_cost_quality_tradeoff(summary, out / 'cost_quality_tradeoff.png', 0.20)
mp.plot_cumulative_loss(traj, out / 'cumulative_loss.png', 0.20)
mp.plot_create_rate_by_cost(summary, out / 'create_rate_by_cost.png')
mp.plot_action_library_growth(traj, out / 'action_library_growth.png', 0.20)
print('saved figures to', out)

In [ ]:
from IPython.display import Image, display
for name in ['cost_quality_tradeoff.png','cumulative_loss.png',
             'create_rate_by_cost.png','action_library_growth.png']:
    display(Image(filename=str(ROOT / 'results' / name)))